In [5]:
import torch
import torch.nn as nn
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from torch.utils.data import DataLoader, TensorDataset

# Cargar el dataset
df = pd.read_csv("insurance.csv")

# Preprocesamiento: codificar variables categóricas
label_encoders = {}
for col in ['sex', 'smoker', 'region']:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

# Normalizar las características
scaler = StandardScaler()
X = df.drop(columns=["charges"])
X = scaler.fit_transform(X)

# Convertir a tensores de PyTorch
X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(df["charges"].values, dtype=torch.float32).view(-1, 1)

# Crear DataLoader para entrenamiento
dataset = TensorDataset(X_tensor, y_tensor)
train_loader = DataLoader(dataset, batch_size=32, shuffle=True)

# Definir el modelo de regresión lineal
class LinearRegression(nn.Module):
    def __init__(self, input_dim):
        super(LinearRegression, self).__init__()
        self.linear = nn.Linear(input_dim, 1)

    def forward(self, x):
        return self.linear(x)

# Inicializar modelo, función de pérdida y optimizador
input_size = X.shape[1]
model = LinearRegression(input_size)
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

# Entrenamiento del modelo
num_epochs = 1000
for epoch in range(num_epochs):
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        y_pred = model(X_batch)
        loss = criterion(y_pred, y_batch)
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch}, Loss: {loss.item()}")

    if loss.item() <= 19_000_000:
        print(f"Training stopped at epoch {epoch} with loss: {loss.item()}")
        break


Epoch 0, Loss: 141061728.0
Epoch 1, Loss: 27114804.0
Epoch 2, Loss: 36767736.0
Epoch 3, Loss: 30862698.0
Epoch 4, Loss: 21435346.0
Epoch 5, Loss: 32145980.0
Epoch 6, Loss: 47966300.0
Epoch 7, Loss: 73310096.0
Epoch 8, Loss: 39161864.0
Epoch 9, Loss: 40789276.0
Epoch 10, Loss: 38941528.0
Epoch 11, Loss: 48942076.0
Epoch 12, Loss: 80851824.0
Epoch 13, Loss: 53284540.0
Epoch 14, Loss: 18392030.0
Training stopped at epoch 14 with loss: 18392030.0
